# Experiment 1: LDA topic modeling

This notebook runs a LDA pipeline using preprocessed input from the data preprocessing stage.  

## Hypothesis：

H1 - Bag-of-Words ：In LDA analysis, word order is not important. "network problem" and "problem network" are considered equivalent.  
H2 - Bag-of-words representation is sufficient to capture the all thematic differences between different question types in customer service tickets.  
H3 - The lack of word order and semantic context information can cause LDA to be unable to effectively distinguish semantically similar work order types. Such as "software bug" and "hardware issue"

Also, We made some reasonable hypothesis in the evaluation section to simplify our evaluation process:  

H4 - Coherence is equivalent to quality: We use c_v coherence to select the optimal `n_topics` in evaluate sections, which means a semantically coherent topic = a good topic  
H5 - Subjective setting of λ in `Penalized Score`:  This value was set subjectively in this experiment, but in actual business operations, it can be set using a post-hoc correction method.

## Pipeline: 
**load data - CountVectorizer - LDA - topic export**  

In [7]:
from pathlib import Path
import sys
import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [8]:
import importlib

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

axis2_dir = PROJECT_ROOT / 'notebooks' / '03_Topic_and_Insights'
if str(axis2_dir) not in sys.path:
    sys.path.append(str(axis2_dir))

from Data_preprocessing import Parameters_Path as config
importlib.reload(config)

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_input.csv'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'Experiment_1_LDA'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LDA_TOPICS_PATH = RESULTS_DIR / 'Experiment_1_lda_topics.json'
DOC_TOPIC_PATH = RESULTS_DIR / 'Experiment_1_lda_doc_topics.csv'

count_cfg = config.PARAMETERS['count_vectorizer']
lda_cfg = config.PARAMETERS['lda']
N_TOPICS = int(lda_cfg['n_topics'])
N_TOP_WORDS = int(lda_cfg['n_top_words'])

In [9]:
df = pd.read_csv(INPUT_PATH)
texts = df['text_cleaned_axis1'].astype(str).str.strip()
texts = texts[texts.ne('')]

In [10]:
vectorizer = CountVectorizer(
    max_df=count_cfg['max_df'],
    min_df=count_cfg['min_df'],
    max_features=count_cfg['max_features'],
    stop_words=count_cfg['stop_words']
)
X_count = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=lda_cfg['random_state'],
    learning_method=lda_cfg['learning_method'],
    max_iter=lda_cfg['max_iter']
)
doc_topic = lda.fit_transform(X_count)

print('CountVectorizer config:', count_cfg)
print('LDA config:', lda_cfg)

CountVectorizer config: {'max_df': 0.8, 'min_df': 2, 'max_features': 1200, 'stop_words': 'english'}
LDA config: {'n_topics': 8, 'n_top_words': 8, 'random_state': 42, 'learning_method': 'online', 'max_iter': 10}


In [13]:
def extract_topics(model, vocab, n_top_words=8):
    topics = {}
    for i, topic in enumerate(model.components_, start=1):
        idx = topic.argsort()[-n_top_words:][::-1]
        topics[f'topic_{i}'] = {
            'keywords': [vocab[j] for j in idx],
            'weights': topic[idx].tolist()
        }
    return topics

lda_topics = extract_topics(lda, feature_names, N_TOP_WORDS)

with open(LDA_TOPICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(lda_topics, f, indent=2, ensure_ascii=False)

doc_topic_df = pd.DataFrame(doc_topic, columns=[f'lda_topic_{i}' for i in range(1, N_TOPICS + 1)])
doc_topic_df['max_topic_prob'] = doc_topic_df[[f'lda_topic_{i}' for i in range(1, N_TOPICS + 1)]].max(axis=1)
doc_topic_df['lda_dominant_topic'] = doc_topic_df.values.argmax(axis=1) + 1
doc_topic_df.to_csv(DOC_TOPIC_PATH, index=False, encoding='utf-8')

In [12]:
for topic_name, content in lda_topics.items():
    print(f"{topic_name}: {', '.join(content['keywords'])}")

topic_1: user, inquiry, multiple, contacted, remains, unresolved, mentioned, troubleshooting
topic_2: option, fine, account, unable, facing, acts, unexpectedly, intermittent
topic_3: data, application, specific, loss, deleted, hardware, problems, inquiry
topic_4: screen, message, error, says, mean, peculiar, popping, original
topic_5: account, access, possible, soon, updated, assistance, affecting, related
topic_6: device, account, software, inquiry, changes, data, started, recent
topic_7: resolve, reset, factory, hoping, performed, loss, data, inquiry
topic_8: life, battery, software, tried, available, inquiry, application, updates
